# Re-run `exp_0073` (ST3-enhanced) and save dev logits + confusion matrix

This notebook retrains the submitted ST3-enhanced RoBERTa-large run and, with the
updated `experiments/exp_0073/train.py`, writes three artefacts into
`experiments/exp_0073/`:

- `dev_logits.npz`  — per-example dev logits (ids, label_ids, logits, label_order)
- `confusion_matrix.json` — labelled confusion matrix (rows=true, cols=pred)
- `metrics.json` — now also carries the confusion matrix inline

**Runtime:** set **Runtime -> Change runtime type -> GPU (T4)**, then **Runtime -> Run all**.
The run takes only a few minutes at this corpus scale. Download the three artefacts at
the end and drop them into your local `experiments/exp_0073/` so the confusion matrix
can be folded into paper sections 4.4 / 5.3.

In [ ]:
!nvidia-smi -L   # confirm a GPU is attached; if this errors, switch the runtime to GPU

In [ ]:
# Clone the repo (data, synth_v002_st3 and pseudo_v001_st3 are all committed).
# If you keep the repo on Drive instead, mount it and %cd there rather than cloning.
!git clone https://github.com/psychias/fallacy_detection_clef_2026.git
%cd fallacy_detection_clef_2026
!pip install -q -r requirements.txt

In [ ]:
# Retrain exp_0073. run_experiment.py sets FALLACY_WORKSPACE to the repo root so
# shared/data_utils.py locates data/ and data_synth/ automatically.
!python scripts/run_experiment.py exp_0073

In [ ]:
# Inspect the artefacts.
import json, numpy as np
cm = json.load(open('experiments/exp_0073/confusion_matrix.json'))
print('labels (rows=true, cols=pred):', cm['labels'])
for lbl, row in zip(cm['labels'], cm['matrix']):
    print(f'{lbl:22s} {row}')
z = np.load('experiments/exp_0073/dev_logits.npz', allow_pickle=True)
print('\ndev_logits.npz:', {k: z[k].shape for k in z.files})
print('n dev examples:', z['logits'].shape[0])

In [ ]:
# Download the three artefacts to fold back into the local repo.
from google.colab import files
for f in ('confusion_matrix.json', 'dev_logits.npz', 'metrics.json'):
    files.download(f'experiments/exp_0073/{f}')